<a href="https://colab.research.google.com/github/AzizulHakim00/Glaucomma/blob/main/Glaucomma_RimGraphDG_Single_Cell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# RimGraph-DG V4.3 — select a T4 GPU runtime and run this ONE cell.
# Prototype-active micro-batches, full-stage checkpoint guards, corrected vCDR, and autograd-safe checkpointing.
GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': '',
    'fast_dev_run': False,
    'run_name': 'paper_run_v43',
    'code_revision': 'rimgraph-dg-v4.3-20260804',
    'seeds': [2029],
    'run_global_baseline': True,
    'run_full_model': True,
    'run_optuna': False,
    'optuna_trials': 8,
    'image_size': 320,
    'batch_size': 2,
    'grad_accum': 2,
    'fpn_dim': 128,
    'gradient_checkpointing': True,
    'baseline_epochs': 12,
    'full_epochs': 30,
    'seg_warmup_epochs': 5,
    'anatomy_warmup_epochs': 10,
    'num_workers': 0,
    'n_visual_examples': 2,
}

import hashlib
import json
import traceback
import urllib.request
from pathlib import Path

COMMIT = '76a8712b74e0a998b2b671610a2dc56ad822a811'
EXPECTED_RAW_SHA256 = '46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2'
ROOT = f'https://raw.githubusercontent.com/AzizulHakim00/Glaucomma/{COMMIT}'
parts = [f'v4_parts/part_{i:02d}.py' for i in range(7)]
raw_code = '\n'.join(
    urllib.request.urlopen(f'{ROOT}/{name}').read().decode('utf-8')
    for name in parts
)
actual_raw = hashlib.sha256(raw_code.encode('utf-8')).hexdigest()
assert actual_raw == EXPECTED_RAW_SHA256, f'V4 raw runner integrity check failed: {actual_raw}'

patch_specs = [
    ('runner_patch_v41.py', 'apply_v41'),
    ('runner_patch_v42.py', 'apply_v42'),
    ('runner_patch_v43.py', 'apply_v43'),
    ('runner_patch_v43_autograd.py', 'apply_v43_autograd'),
]
code = raw_code
for patch_name, function_name in patch_specs:
    source = urllib.request.urlopen(f'{ROOT}/{patch_name}').read().decode('utf-8')
    namespace = {}
    exec(compile(source, patch_name, 'exec'), namespace, namespace)
    code = namespace[function_name](code)
compile(code, 'rimgraph_dg_v43_single_cell.py', 'exec')

try:
    exec(code, globals(), globals())
except BaseException:
    trace = traceback.format_exc()
    print(trace)
    local_failure = Path('/content/RimGraph_V43_FAILURE_TRACEBACK.txt')
    try:
        local_failure.write_text(trace, encoding='utf-8')
    except Exception:
        pass
    try:
        failure_dir = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v43')
        failure_dir.mkdir(parents=True, exist_ok=True)
        (failure_dir / 'FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
        (failure_dir / 'FAILURE_STATUS.json').write_text(
            json.dumps({'status': 'failed', 'traceback_file': str(failure_dir / 'FAILURE_TRACEBACK.txt')}, indent=2),
            encoding='utf-8',
        )
    except Exception as drive_error:
        print(f'Could not persist failure to Drive: {drive_error}')
    raise


Mounted at /content/drive
Resolved Colab output: /content/Glaucomma_runs/paper_run_v43
Resolved Drive output: /content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v43
Drive write verification: PASSED


## RimGraph-DG V4 configuration

,setting,value
0,kaggle_dataset,arnavjain1/glaucoma-datasets
1,manual_data_dir,
2,sources,"['ORIGA', 'REFUGE', 'G1020']"
3,fold_targets,"['ORIGA', 'REFUGE', 'G1020']"
4,image_size,320
5,num_workers,0
6,canonicalize_laterality,True
7,exclude_cross_source_duplicates,True
8,project_name,RimGraph_DG_V4
9,run_name,paper_run_v43


## Downloading or locating Kaggle dataset

100%|██████████| 5.55G/5.55G [01:05<00:00, 91.4MB/s]

...

## Dataset audit

label,source,Normal,Glaucoma,Total labelled,With any mask,Known laterality,Excluded unlabeled
0,G1020,724,296,1020,1020,0,0
1,ORIGA,482,168,650,650,650,0
2,REFUGE,720,80,800,800,0,400


## Seed 2029 — held-out ORIGA

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            